# ROGII Wellbore Geology Prediction v2-fast Submission

Memory-safe experimental Kaggle notebook. It reads the competition data from `/kaggle/input/`, builds a compact train/test-compatible tabular feature set, trains a one-seed LightGBM model when available, validates the submission schema, and writes `/kaggle/working/submission.csv`.

This notebook intentionally avoids the full v2 feature explosion that caused a Kaggle out-of-memory restart.


In [ ]:
from __future__ import annotations

import gc
import os
from pathlib import Path

import numpy as np
import pandas as pd


SEED = 2026
TARGET = "TVT"
SUBMISSION_TARGET = "tvt"
KAGGLE_INPUT_ROOT = Path("/kaggle/input")
PREFERRED_INPUT_DIR = KAGGLE_INPUT_ROOT / "rogii-wellbore-geology-prediction"
OUTPUT_PATH = Path("/kaggle/working/submission.csv")

RUN_LOCAL_VALIDATION = False
VALIDATION_MAX_WELLS = 60
CLIP_PREDICTIONS = True
CLIP_QUANTILES = (0.001, 0.999)
SMOOTH_PREDICTIONS = False
SMOOTH_WINDOW = 5
SMOOTH_BLEND = 0.10

HORIZONTAL_FEATURE_COLS = ["MD", "X", "Y", "Z", "GR", "TVT_input"]
TYPEWELL_NUMERIC_COLS = ["TVT", "GR"]
FLOAT_COLS = HORIZONTAL_FEATURE_COLS + [TARGET]
TRAIN_ONLY_HORIZONTAL_COLS = {"ANCC", "ASTNU", "ASTNL", "EGFDU", "EGFDL", "BUDA"}


def well_id_from_path(path: Path) -> str:
    return path.name.split("__", 1)[0].split(".", 1)[0]


def process_memory_mb() -> float:
    try:
        import psutil

        return float(psutil.Process(os.getpid()).memory_info().rss / 1024**2)
    except Exception:
        try:
            # Linux returns KB, macOS returns bytes. Kaggle is Linux.
            import resource

            rss = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
            return float(rss / 1024.0)
        except Exception:
            return float("nan")


def frame_memory_mb(df: pd.DataFrame | None) -> float:
    if df is None:
        return 0.0
    return float(df.memory_usage(deep=True).sum() / 1024**2)


def log_memory(stage: str, **frames: pd.DataFrame) -> None:
    parts = [f"{name}={frame_memory_mb(frame):.1f} MB" for name, frame in frames.items()]
    detail = ", ".join(parts)
    if detail:
        detail = f" | {detail}"
    print(f"[memory] {stage}: process_rss={process_memory_mb():.1f} MB{detail}")


def find_input_dir() -> Path:
    candidates: list[Path] = []
    if (PREFERRED_INPUT_DIR / "sample_submission.csv").exists():
        candidates.append(PREFERRED_INPUT_DIR)
    if KAGGLE_INPUT_ROOT.exists():
        candidates.extend(path.parent for path in sorted(KAGGLE_INPUT_ROOT.rglob("sample_submission.csv")))

    unique_candidates = list(dict.fromkeys(candidates))
    valid_candidates = [path for path in unique_candidates if (path / "train").is_dir() and (path / "test").is_dir()]
    if len(valid_candidates) == 1:
        return valid_candidates[0]
    if len(valid_candidates) > 1:
        names = [str(path) for path in valid_candidates]
        raise ValueError(f"Multiple possible competition input directories found: {names}")

    available = sorted(str(path) for path in KAGGLE_INPUT_ROOT.glob("*")) if KAGGLE_INPUT_ROOT.exists() else []
    nested_samples = sorted(str(path) for path in KAGGLE_INPUT_ROOT.rglob("sample_submission.csv")) if KAGGLE_INPUT_ROOT.exists() else []
    raise FileNotFoundError(
        "Could not find a Kaggle input directory containing sample_submission.csv, train/, and test/. "
        f"Available /kaggle/input entries: {available}. "
        f"Nested sample_submission.csv files found: {nested_samples}"
    )


def list_files(input_dir: Path, split: str, kind: str) -> list[Path]:
    folder = input_dir / split
    patterns = {"horizontal": "*__horizontal_well.csv", "typewell": "*__typewell.csv"}
    if kind not in patterns:
        raise ValueError(f"Unknown file kind: {kind}")
    files = sorted(folder.glob(patterns[kind]))
    if not files:
        raise FileNotFoundError(f"No {kind} files found under {folder}")
    return files


def downcast_numeric(df: pd.DataFrame) -> pd.DataFrame:
    for col in df.columns:
        if pd.api.types.is_float_dtype(df[col]):
            df[col] = pd.to_numeric(df[col], downcast="float")
        elif pd.api.types.is_integer_dtype(df[col]):
            df[col] = pd.to_numeric(df[col], downcast="integer")
    return df


def load_horizontal(input_dir: Path, split: str) -> pd.DataFrame:
    frames: list[pd.DataFrame] = []
    for well_code, path in enumerate(list_files(input_dir, split, "horizontal")):
        allowed = set(HORIZONTAL_FEATURE_COLS)
        if split == "train":
            allowed.add(TARGET)
        dtype = {col: "float32" for col in FLOAT_COLS}
        df = pd.read_csv(
            path,
            usecols=lambda col: col in allowed,
            dtype=dtype,
            low_memory=True,
        )
        ignored = sorted(TRAIN_ONLY_HORIZONTAL_COLS.intersection(set(df.columns)))
        if ignored:
            raise AssertionError(f"Unexpected train-only columns were loaded from {path.name}: {ignored}")
        df.insert(0, "well_id", well_id_from_path(path))
        df.insert(1, "well_code", np.int32(well_code))
        df.insert(2, "row_id", np.arange(len(df), dtype=np.int32))
        frames.append(df)

    out = pd.concat(frames, ignore_index=True, sort=False, copy=False)
    out["well_id"] = out["well_id"].astype("category")
    out["well_code"] = out["well_code"].astype("int32")
    out["row_id"] = out["row_id"].astype("int32")
    out = downcast_numeric(out)
    del frames
    gc.collect()
    return out


def load_typewell_summary(input_dir: Path, split: str) -> pd.DataFrame:
    rows: list[dict[str, float | int | str]] = []
    for path in list_files(input_dir, split, "typewell"):
        df = pd.read_csv(
            path,
            usecols=lambda col: col in TYPEWELL_NUMERIC_COLS,
            dtype={col: "float32" for col in TYPEWELL_NUMERIC_COLS},
            low_memory=True,
        )
        row: dict[str, float | int | str] = {
            "well_id": well_id_from_path(path),
            "typewell_rows": int(len(df)),
        }
        for col in TYPEWELL_NUMERIC_COLS:
            if col not in df.columns:
                continue
            s = df[col]
            prefix = f"typewell_{col}"
            row[f"{prefix}_mean"] = float(s.mean())
            row[f"{prefix}_std"] = float(s.std())
            row[f"{prefix}_min"] = float(s.min())
            row[f"{prefix}_max"] = float(s.max())
            row[f"{prefix}_q50"] = float(s.quantile(0.50))
        rows.append(row)
        del df
    summary = pd.DataFrame(rows)
    if not summary.empty:
        summary["well_id"] = summary["well_id"].astype("category")
        summary = downcast_numeric(summary)
    return summary


def add_basic_features(df: pd.DataFrame, split: str) -> pd.DataFrame:
    work = df.sort_values(["well_code", "row_id"]).reset_index(drop=True)
    group = work.groupby("well_code", sort=False, observed=True)

    feature_blocks: list[pd.DataFrame] = []

    for col in [c for c in HORIZONTAL_FEATURE_COLS if c in work.columns]:
        feature_blocks.append(pd.DataFrame({f"{col}_missing": work[col].isna().astype("int8")}))

    for col in [c for c in ["MD", "X", "Y", "Z", "GR"] if c in work.columns]:
        work[col] = group[col].transform(lambda s: s.ffill().bfill()).astype("float32")
        median = np.float32(work[col].median())
        work[col] = work[col].fillna(median).astype("float32")

    rows_in_well = (group["row_id"].transform("max").astype("float32") + np.float32(1.0)).astype("float32")
    row_id_float = work["row_id"].astype("float32")
    denom = (rows_in_well - np.float32(1.0)).replace(0, 1).astype("float32")
    row_frac = (row_id_float / denom).astype("float32")
    rows_from_end = (rows_in_well - row_id_float - np.float32(1.0)).astype("float32")

    position = pd.DataFrame(
        {
            "rows_in_well": rows_in_well,
            "row_frac": row_frac,
            "rows_from_end": rows_from_end,
        }
    )

    if "TVT_input" in work.columns:
        known = work["TVT_input"].notna()
        known_row = work["row_id"].where(known)
        last_known_row = known_row.groupby(work["well_code"], sort=False).ffill()
        rows_since_last = (work["row_id"].astype("float32") - last_known_row.astype("float32")).fillna(0).astype("float32")
        hidden_denom = (rows_in_well - last_known_row.astype("float32")).replace(0, np.nan).astype("float32")
        hidden_position = (rows_since_last / hidden_denom).replace([np.inf, -np.inf], np.nan).fillna(0).clip(0, 1).astype("float32")
        if "MD" in work.columns:
            known_md = work["MD"].where(known)
            last_known_md = known_md.groupby(work["well_code"], sort=False).ffill()
            md_since_last = (work["MD"] - last_known_md).fillna(0).astype("float32")
        else:
            md_since_last = pd.Series(np.zeros(len(work), dtype=np.float32), index=work.index)
        position["tvt_input_missing"] = (~known).astype("int8")
        position["rows_since_last_tvt_input"] = rows_since_last
        position["hidden_context_position"] = hidden_position
        position["md_since_last_tvt_input"] = md_since_last
    else:
        position["tvt_input_missing"] = np.int8(0)
        position["rows_since_last_tvt_input"] = np.float32(0.0)
        position["hidden_context_position"] = np.float32(0.0)
        position["md_since_last_tvt_input"] = np.float32(0.0)

    feature_blocks.append(position)

    del rows_in_well, row_id_float, denom, row_frac, rows_from_end, position
    gc.collect()

    delta_cols: dict[str, pd.Series] = {}
    for col in [c for c in ["MD", "X", "Y", "Z", "GR"] if c in work.columns]:
        delta_cols[f"{col}_diff1"] = group[col].diff().fillna(0).astype("float32")
    deltas = pd.DataFrame(delta_cols)
    feature_blocks.append(deltas)

    md_diff = deltas["MD_diff1"].replace(0, np.nan).astype("float32") if "MD_diff1" in deltas else None
    slope_cols: dict[str, pd.Series] = {}
    if md_diff is not None:
        for col in ["X", "Y", "Z", "GR"]:
            diff_col = f"{col}_diff1"
            if diff_col in deltas:
                slope_cols[f"{col}_slope_per_md"] = (deltas[diff_col] / md_diff).replace([np.inf, -np.inf], np.nan).fillna(0).astype("float32")
    if slope_cols:
        feature_blocks.append(pd.DataFrame(slope_cols))

    if {"X_diff1", "Y_diff1", "Z_diff1"}.issubset(deltas.columns):
        step_distance_xy = np.sqrt(deltas["X_diff1"] ** 2 + deltas["Y_diff1"] ** 2).astype("float32")
        step_distance_3d = np.sqrt(step_distance_xy**2 + deltas["Z_diff1"] ** 2).astype("float32")
        distance = pd.DataFrame(
            {
                "step_distance_xy": step_distance_xy,
                "step_distance_3d": step_distance_3d,
                "vertical_delta_abs": deltas["Z_diff1"].abs().astype("float32"),
            }
        )
        if md_diff is not None:
            distance["step_distance_3d_per_md"] = (step_distance_3d / md_diff).replace([np.inf, -np.inf], np.nan).fillna(0).astype("float32")
        feature_blocks.append(distance)
        work["step_distance_3d"] = step_distance_3d
        del distance, step_distance_xy, step_distance_3d

    rolling_cols: dict[str, pd.Series] = {}
    if "GR" in work.columns:
        rolling = group["GR"].rolling(7, min_periods=1)
        rolling_cols["GR_roll7_mean"] = rolling.mean().reset_index(level=0, drop=True).astype("float32")
        rolling_cols["GR_roll7_std"] = rolling.std().reset_index(level=0, drop=True).fillna(0).astype("float32")
    if rolling_cols:
        feature_blocks.append(pd.DataFrame(rolling_cols))

    aggregate_cols: dict[str, pd.Series] = {}
    for col in [c for c in ["MD", "GR", "step_distance_3d"] if c in work.columns]:
        col_group = work.groupby("well_code", sort=False, observed=True)[col]
        aggregate_cols[f"well_{col}_mean"] = col_group.transform("mean").astype("float32")
        aggregate_cols[f"well_{col}_std"] = col_group.transform("std").fillna(0).astype("float32")
        aggregate_cols[f"well_{col}_min"] = col_group.transform("min").astype("float32")
        aggregate_cols[f"well_{col}_max"] = col_group.transform("max").astype("float32")
        aggregate_cols[f"well_{col}_range"] = (aggregate_cols[f"well_{col}_max"] - aggregate_cols[f"well_{col}_min"]).astype("float32")
    if "step_distance_3d" in work.columns:
        aggregate_cols["well_path_length_3d"] = work.groupby("well_code", sort=False, observed=True)["step_distance_3d"].transform("sum").astype("float32")
    if aggregate_cols:
        feature_blocks.append(pd.DataFrame(aggregate_cols))

    base_cols = ["well_id", "well_code", "row_id"] + [c for c in ["MD", "X", "Y", "Z", "GR", "TVT_input", TARGET] if c in work.columns]
    out = pd.concat([work[base_cols]] + feature_blocks, axis=1, copy=False)
    out = downcast_numeric(out)

    del work, feature_blocks
    gc.collect()
    print(f"{split} feature columns before typewell merge: {len([c for c in out.columns if c not in {'well_id', TARGET}])}")
    return out


def add_typewell_features(input_dir: Path, df: pd.DataFrame, split: str) -> pd.DataFrame:
    summary = load_typewell_summary(input_dir, split)
    if summary.empty:
        return df
    work = df.merge(summary, on="well_id", how="left", copy=False)
    if "GR" in work.columns and "typewell_GR_mean" in work.columns:
        denom = work["typewell_GR_std"].replace(0, np.nan).astype("float32")
        work["GR_minus_typewell_GR_mean"] = (work["GR"] - work["typewell_GR_mean"]).astype("float32")
        work["GR_minus_typewell_GR_q50"] = (work["GR"] - work["typewell_GR_q50"]).astype("float32")
        work["GR_typewell_z"] = ((work["GR"] - work["typewell_GR_mean"]) / denom).replace([np.inf, -np.inf], np.nan).fillna(0).astype("float32")
    work = downcast_numeric(work)
    del summary
    gc.collect()
    return work


def build_prediction_ids(df: pd.DataFrame) -> pd.Series:
    return df["well_id"].astype(str) + "_" + df["row_id"].astype(str)


def select_features(train: pd.DataFrame, test: pd.DataFrame) -> list[str]:
    excluded = {TARGET, "TVT_input", "well_id", "well_code", "id"}
    features = [
        col
        for col in train.columns
        if col in test.columns
        and col not in excluded
        and pd.api.types.is_numeric_dtype(train[col])
        and pd.api.types.is_numeric_dtype(test[col])
    ]
    if not features:
        raise ValueError("No common numeric features are available.")
    if len(features) > 90:
        raise ValueError(f"v2-fast feature count exceeded target: {len(features)} > 90")
    return features


def clean_matrices(train: pd.DataFrame, test: pd.DataFrame, features: list[str]) -> tuple[pd.DataFrame, pd.DataFrame]:
    train_x = train.loc[:, features].replace([np.inf, -np.inf], np.nan)
    test_x = test.loc[:, features].replace([np.inf, -np.inf], np.nan)
    medians = train_x.median(numeric_only=True).fillna(0).astype("float32")
    train_x = train_x.fillna(medians).fillna(0).astype("float32", copy=False)
    test_x = test_x.fillna(medians).fillna(0).astype("float32", copy=False)
    return train_x, test_x


def rmse(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    return float(np.sqrt(np.mean((np.asarray(y_true, dtype=np.float32) - np.asarray(y_pred, dtype=np.float32)) ** 2)))


def maybe_validate(train_x: pd.DataFrame, y: np.ndarray, groups: pd.Series) -> None:
    if not RUN_LOCAL_VALIDATION:
        print("Local validation disabled by default for Kaggle memory safety.")
        return

    from sklearn.model_selection import GroupShuffleSplit

    unique_groups = pd.Series(groups.unique()).sample(
        n=min(VALIDATION_MAX_WELLS, groups.nunique()),
        random_state=SEED,
    )
    mask = groups.isin(unique_groups).to_numpy()
    x_small = train_x.loc[mask]
    y_small = y[mask]
    groups_small = groups.loc[mask]
    splitter = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=SEED)
    fit_idx, valid_idx = next(splitter.split(x_small, y_small, groups_small))
    model_name, model = build_model()
    model.fit(x_small.iloc[fit_idx], y_small[fit_idx])
    pred = np.asarray(model.predict(x_small.iloc[valid_idx]), dtype=np.float32)
    print(f"Sampled validation model={model_name}, wells={groups_small.nunique()}, RMSE={rmse(y_small[valid_idx], pred):.6f}")
    del x_small, y_small, groups_small, pred, model
    gc.collect()


def build_model() -> tuple[str, object]:
    try:
        from lightgbm import LGBMRegressor

        return "lightgbm_v2_fast", LGBMRegressor(
            objective="regression",
            n_estimators=450,
            learning_rate=0.045,
            num_leaves=48,
            max_depth=8,
            min_child_samples=80,
            subsample=0.85,
            colsample_bytree=0.82,
            reg_alpha=0.05,
            reg_lambda=0.20,
            random_state=SEED,
            n_jobs=4,
            verbose=-1,
        )
    except Exception as lightgbm_error:
        from sklearn.ensemble import HistGradientBoostingRegressor

        print(f"LightGBM unavailable; using HistGradientBoostingRegressor. Error: {lightgbm_error}")
        return "hist_gradient_boosting_v2_fast", HistGradientBoostingRegressor(
            max_iter=350,
            learning_rate=0.045,
            max_leaf_nodes=48,
            n_iter_no_change=20,
            l2_regularization=0.02,
            random_state=SEED,
        )


def clip_predictions(pred: np.ndarray, y: np.ndarray) -> np.ndarray:
    if not CLIP_PREDICTIONS:
        print("Prediction clipping disabled.")
        return pred
    low, high = np.quantile(y, CLIP_QUANTILES)
    print(f"Prediction clipping enabled: quantiles={CLIP_QUANTILES}, bounds=({low:.4f}, {high:.4f})")
    return np.clip(pred, low, high).astype("float32")


def maybe_smooth_submission(submission: pd.DataFrame) -> pd.DataFrame:
    if not SMOOTH_PREDICTIONS:
        print("Prediction smoothing disabled by default.")
        return submission

    work = submission.copy()
    parsed = work["id"].astype(str).str.rsplit("_", n=1, expand=True)
    work["_well_id"] = parsed[0]
    work["_row_id"] = pd.to_numeric(parsed[1], errors="coerce").astype("int32")
    work["_order"] = np.arange(len(work), dtype=np.int32)
    work = work.sort_values(["_well_id", "_row_id"])
    smoothed = (
        work.groupby("_well_id", sort=False)[SUBMISSION_TARGET]
        .transform(lambda s: s.rolling(SMOOTH_WINDOW, min_periods=1, center=True).mean())
        .astype("float32")
    )
    work[SUBMISSION_TARGET] = ((1.0 - SMOOTH_BLEND) * work[SUBMISSION_TARGET] + SMOOTH_BLEND * smoothed).astype("float32")
    work = work.sort_values("_order").drop(columns=["_well_id", "_row_id", "_order"])
    print(f"Prediction smoothing enabled: window={SMOOTH_WINDOW}, blend={SMOOTH_BLEND}")
    return work


def validate_submission(output_path: Path, sample: pd.DataFrame) -> None:
    if not output_path.exists():
        raise FileNotFoundError(f"Submission file was not written: {output_path}")

    sub = pd.read_csv(output_path)
    problems: list[str] = []
    if list(sub.columns) != list(sample.columns):
        problems.append(f"columns {list(sub.columns)} do not match {list(sample.columns)}")
    if SUBMISSION_TARGET not in sub.columns:
        problems.append(f"prediction column must be named {SUBMISSION_TARGET!r}")
    if len(sub) != len(sample):
        problems.append(f"row count {len(sub)} does not match {len(sample)}")
    if "id" in sub.columns and not sub["id"].equals(sample["id"]):
        problems.append("id order does not match sample_submission.csv")
    if SUBMISSION_TARGET in sub.columns:
        values = sub[SUBMISSION_TARGET].to_numpy(dtype=np.float64)
        if not np.isfinite(values).all():
            problems.append("submission contains NaN or infinite predictions")

    if problems:
        raise ValueError(f"Submission validation failed: {problems}")


def main() -> None:
    input_dir = find_input_dir()
    print(f"Using competition input directory: {input_dir}")

    sample_path = input_dir / "sample_submission.csv"
    sample = pd.read_csv(sample_path, low_memory=True)
    if list(sample.columns) != ["id", SUBMISSION_TARGET]:
        raise ValueError(f"Expected sample submission columns ['id', '{SUBMISSION_TARGET}'], got {list(sample.columns)}")

    raw_train = load_horizontal(input_dir, "train")
    log_memory("after train load", raw_train=raw_train)
    raw_test = load_horizontal(input_dir, "test")
    log_memory("after test load", raw_train=raw_train, raw_test=raw_test)

    train = add_basic_features(raw_train, "train")
    del raw_train
    gc.collect()
    train = add_typewell_features(input_dir, train, "train")
    log_memory("after train features", train=train)

    test = add_basic_features(raw_test, "test")
    del raw_test
    gc.collect()
    test = add_typewell_features(input_dir, test, "test")
    log_memory("after test features", train=train, test=test)

    if TARGET not in train.columns:
        raise ValueError(f"Training data does not contain target column {TARGET!r}")
    train = train.loc[train[TARGET].notna()].reset_index(drop=True)
    y = train[TARGET].to_numpy(dtype=np.float32)

    test["id"] = build_prediction_ids(test)
    features = select_features(train, test)
    print(f"v2-fast selected feature count: {len(features)}")
    print(f"Features: {features}")

    train_x, test_x = clean_matrices(train, test, features)
    groups = train["well_id"].copy()
    log_memory("after matrix cleanup", train_x=train_x, test_x=test_x)

    maybe_validate(train_x, y, groups)
    del groups
    gc.collect()

    model_name, model = build_model()
    print(f"Training model: {model_name}")
    model.fit(train_x, y)
    predictions = np.asarray(model.predict(test_x), dtype=np.float32)
    if not np.isfinite(predictions).all():
        raise ValueError("Model produced non-finite test predictions.")

    predictions = clip_predictions(predictions, y)
    del train_x, y, model
    gc.collect()

    pred = pd.DataFrame({"id": test["id"].to_numpy(), SUBMISSION_TARGET: predictions})
    submission = sample[["id"]].merge(pred, on="id", how="left", validate="one_to_one")
    if submission[SUBMISSION_TARGET].isna().any():
        missing_ids = submission.loc[submission[SUBMISSION_TARGET].isna(), "id"].head(10).tolist()
        raise ValueError(f"Could not generate predictions for all sample IDs. First missing IDs: {missing_ids}")

    submission[SUBMISSION_TARGET] = submission[SUBMISSION_TARGET].astype("float32")
    submission = maybe_smooth_submission(submission)
    submission = submission[list(sample.columns)]

    OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
    submission.to_csv(OUTPUT_PATH, index=False)
    validate_submission(OUTPUT_PATH, sample)
    log_memory("after submission creation", submission=submission)

    print("Kaggle v2-fast submission completed.")
    print(f"Model: {model_name}")
    print(f"Feature count: {len(features)}")
    print(f"Submission rows: {len(submission)}")
    print(f"Prediction range: {submission[SUBMISSION_TARGET].min():.4f} to {submission[SUBMISSION_TARGET].max():.4f}")
    print(f"Wrote: {OUTPUT_PATH}")
    print("Validation passed: columns, row count, id order, prediction column tvt, and finite predictions.")


if __name__ == "__main__":
    main()
